# Marketing Science Paper — Empirical Tables, AMCE / WTP Recovery, & Ablation Figures

Pure CPU analysis notebook (no GPU required) reading saved runs from `ms_paper_beta_loss/runs/{kev-0.8b,kev-4b}/`:

1. **Run Inventory**: Status and raw metrics across all regimes on `kev-0.8b` and `kev-4b`.
2. **Table 1 (Main Paper Table)**: 5-row information ladder (`Untrained Base` $\to$ `Table Only (fitted OLS)` $\to$ `In-House Microdata Only` $\to$ `Hybrid Microdata + Table` $\to$ `Full-Microdata Reference`) with `% of Row 5's MSE reduction recovered` (can exceed 100%).
3. **Table 2 (Single-Study Controlled Comparison)**: `wu2025` 8 fitted OLS slopes vs the `wu2025` 18,591 respondent choices they were fitted from.
4. **Table 3 (Econometric Attribute-Level AMCE & Implied WTP in USD)**: Unseen `kriner2026` flight conjoint attribute slopes and dollar Willingness-To-Pay ($\text{WTP}_k = -\beta_k / \beta_{\text{price per \$}}$), plus `grummon2025` food-label slopes.
5. **Figure 1 (KL Anchor Ablation)**: Optimization dynamics of `row2_wu_table_anchored` ($\alpha = 0.02$) vs `row2_ablation_wu_table_no_kl` ($\alpha = 0.0$).
6. **Reference Cross-Check (`v8`, `v12`, `v13`, `v14`)**: Explicit baseline table from pre-paper runs in `beta-training/runs/`.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (c for p in (Path.cwd(), *Path.cwd().parents) for c in (p, p / "code") if all((c / d).is_dir() for d in ("kev", "data", "training"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and training/ at or above {Path.cwd()}")

BETA_TRAINING_DIR = CODE_DIR / "train-notebooks" / "beta-training"
PAPER_DIR = BETA_TRAINING_DIR / "ms_paper_beta_loss"
PAPER_RUNS_ROOT = PAPER_DIR / "runs"
LEGACY_RUNS_ROOT = BETA_TRAINING_DIR / "runs"

os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(BETA_TRAINING_DIR), str(PAPER_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from beta_ce_population.trainer import load_ce_population_run
from beta_population.trainer import load_population_run
from beta_train import runs as run_store
from beta_train.trainer import load_run
from paper_beta import (
    extract_step0_from_run_dir,
    load_paper_run,
    saved_run_dirs,
    summarize_paper_runs_dir,
)

# Models to analyse: each needs at least one saved paper run
MODEL_SLUGS = ("kev-0.8b", "kev-4b")
VALIDATORS = ("kriner2026", "grummon2025")
SUMMARY_METRICS = ("beta_mse", "std_beta_mse", "beta_mae", "spearman_rho", "sign_acc", "kl_loss")
# Table 1 rows after Row 1 (step 0)
MAIN_TABLE_ORDER = (
    "row2_wu_table_anchored",
    "row3_berke_ce_only",
    "row4_hybrid_berke_ce_wu_table",
    "row5_full_berke_wu_ce_oracle",
)
print(f"PAPER_RUNS_ROOT : {PAPER_RUNS_ROOT}")
print(f"LEGACY_RUNS_ROOT: {LEGACY_RUNS_ROOT}")

## 1. Paper Run Inventory (`ms_paper_beta_loss/runs/`)

In [ ]:
inventory_by_model: dict[str, pd.DataFrame] = {}
for model_slug in MODEL_SLUGS:
    model_runs_dir = PAPER_RUNS_ROOT / model_slug
    summary_df = summarize_paper_runs_dir(model_runs_dir, VALIDATORS, SUMMARY_METRICS)
    inventory_by_model[model_slug] = summary_df
    print(f"=== {model_slug} ({model_runs_dir}) ===")
    display(summary_df.round(5))

## 2. Table 1 (Main 5-Row Ladder) & Table 2 (Single-Study Controlled Comparison)

For each model (`kev-0.8b`, `kev-4b`) with saved paper runs, builds:
- **Table 1**: `Row 1 (Base)` $\to$ `Row 2 (Wu Table Only)` $\to$ `Row 3 (Berke Microdata Only)` $\to$ `Row 4 (Hybrid Berke y + Wu Table)` $\to$ `Row 5 (Full Wu + Berke Microdata Reference)`, plus `% of Row 5's MSE reduction recovered` (can exceed 100%; Table 2 uses Row 2b as the reference).
- **Table 2**: `Row 1 (Base)` vs `Row 2 (Wu Table Only, 8 numbers)` vs `Row 2b (Wu Microdata Only, 18,591 choices)`.

In [ ]:
def build_ladder_table(summary_df: pd.DataFrame, row_slugs: tuple[str, ...], reference_slug: str) -> pd.DataFrame:
    # Rows that are not finished keep their status and NaN metrics; rows with no run folder yet are "not_run_yet"
    ordered = ["row1_base_untrained", *row_slugs]
    sub = summary_df.reindex(ordered)
    sub["status"] = sub["status"].fillna("not_run_yet")
    reference_row = sub.loc[reference_slug]
    if reference_row["status"] != "finished":
        raise ValueError(f"{reference_slug} is {reference_row['status']}; the recovery column needs it finished")
    base_row = sub.loc["row1_base_untrained"]
    out_cols = ["row", "status", "best_step"]
    for v in VALIDATORS:
        mse_col = f"{v}/beta_mse"
        rec_col = f"{v}/pct_of_reference_reduction"
        base_mse = float(base_row[mse_col])
        reference_mse = float(reference_row[mse_col])
        reference_reduction = base_mse - reference_mse
        if reference_reduction <= 0:
            raise ValueError(
                f"{v}: {reference_slug} beta MSE {reference_mse:.5f} is not below the base model's {base_mse:.5f}"
            )
        sub[rec_col] = 100.0 * (base_mse - sub[mse_col].astype(float)) / reference_reduction
        out_cols.extend(
            [
                mse_col,
                rec_col,
                f"{v}/beta_mae",
                f"{v}/spearman_rho",
                f"{v}/sign_acc",
                f"{v}/kl_loss",
            ]
        )
    return sub[out_cols]


for model_slug, summary_df in inventory_by_model.items():
    print(f"=== Table 1 (Main 5-Row Paper Table) — {model_slug} ===")
    table_1 = build_ladder_table(summary_df, MAIN_TABLE_ORDER, reference_slug="row5_full_berke_wu_ce_oracle")
    display(table_1.round(5))

    print(f"=== Table 2 (Single-Study Controlled Comparison: Wu Table vs Wu Microdata) — {model_slug} ===")
    table_2 = build_ladder_table(
        summary_df,
        ("row2_wu_table_anchored", "row2_ablation_wu_table_no_kl", "row2b_wu_ce_oracle"),
        reference_slug="row2b_wu_ce_oracle",
    )
    display(table_2.round(5))

## 3. Table 3 — Econometric Attribute-Level AMCEs & Implied Flight WTP (USD)

On unseen `kriner2026`, flight ticket prices vary across four discrete levels (`$250` reference, `$275`, `$300`, `$325`). Fitting the linear price gradient $\beta_{\text{price per \$}}$ across $\Delta p \in \{0, 25, 50, 75\}$ yields each model's implied dollar Willingness-To-Pay ($\text{WTP}_k = -\beta_k / \beta_{\text{price per \$}}$) for every non-price attribute.

In [ ]:
KRINER_PRICE_LEVELS = {
    "Price: $275": 25.0,
    "Price: $300": 50.0,
    "Price: $325": 75.0,
}


def kriner_wtp_from_series(beta_series: pd.Series) -> pd.Series:
    """Compute per-dollar price slope (OLS slope with intercept across price changes [0, 25, 50, 75]) and implied USD WTP."""
    deltas = np.array([0.0, 25.0, 50.0, 75.0], dtype=np.float64)
    effects = np.array(
        [
            0.0,
            float(beta_series["Price: $275"]),
            float(beta_series["Price: $300"]),
            float(beta_series["Price: $325"]),
        ],
        dtype=np.float64,
    )
    centered_x = deltas - deltas.mean()
    centered_y = effects - effects.mean()
    slope_per_dollar = float((centered_x @ centered_y) / (centered_x @ centered_x))
    if slope_per_dollar >= 0:
        raise ValueError(f"Non-negative price slope {slope_per_dollar:.6f}; dollar WTP is ill-defined")
    non_price_terms = [t for t in beta_series.index if t not in KRINER_PRICE_LEVELS]
    wtp = {term: -float(beta_series[term]) / slope_per_dollar for term in non_price_terms}
    wtp["Marginal Price Slope (dPr / $100)"] = slope_per_dollar * 100.0
    return pd.Series(wtp)


def require_term_index(beta_df: pd.DataFrame) -> pd.DataFrame:
    # Saved and in-memory beta tables are indexed by term
    if beta_df.index.name != "term":
        raise ValueError(f"Expected a beta table indexed by 'term', got index {beta_df.index.name!r}")
    return beta_df


def model_betas_in_order(beta_df: pd.DataFrame, human_index: pd.Index, label: str) -> pd.Series:
    model_betas = require_term_index(beta_df)["model_pop_beta"]
    if set(model_betas.index) != set(human_index):
        raise ValueError(f"{label}: terms {sorted(model_betas.index)} != human terms {sorted(human_index)}")
    return model_betas.loc[human_index]


def build_amce_and_wtp_tables(named_beta_dfs: dict[str, dict[str, pd.DataFrame]]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Return (kriner_amce_df, kriner_wtp_usd_df, grummon_amce_df) across named models."""
    first_entry = next(iter(named_beta_dfs.values()))
    kriner_human = require_term_index(first_entry["kriner2026"])["human_pop_beta"]
    grummon_human = require_term_index(first_entry["grummon2025"])["human_pop_beta"]

    kriner_cols = {"Human (fitted OLS)": kriner_human}
    grummon_cols = {"Human (fitted OLS)": grummon_human}
    for label, by_val in named_beta_dfs.items():
        kriner_cols[label] = model_betas_in_order(by_val["kriner2026"], kriner_human.index, label)
        grummon_cols[label] = model_betas_in_order(by_val["grummon2025"], grummon_human.index, label)

    kriner_amce_df = pd.DataFrame(kriner_cols)
    grummon_amce_df = pd.DataFrame(grummon_cols)

    wtp_cols = {}
    for col_name in kriner_amce_df.columns:
        wtp_cols[col_name] = kriner_wtp_from_series(kriner_amce_df[col_name])
    kriner_wtp_df = pd.DataFrame(wtp_cols)
    wtp_errors = kriner_wtp_df.iloc[:-1].sub(kriner_wtp_df["Human (fitted OLS)"].iloc[:-1], axis=0)
    kriner_wtp_df.loc["MAE vs Human WTP ($)"] = wtp_errors.abs().mean()
    return kriner_amce_df, kriner_wtp_df, grummon_amce_df


for model_slug in MODEL_SLUGS:
    model_runs_dir = PAPER_RUNS_ROOT / model_slug
    summary_df = inventory_by_model[model_slug]
    named_betas: dict[str, dict[str, pd.DataFrame]] = {}
    step0_info = extract_step0_from_run_dir(saved_run_dirs(model_runs_dir)[0], VALIDATORS)
    named_betas["Row 1: Base LLM"] = step0_info["betas_by_validator"]
    for slug in MAIN_TABLE_ORDER:
        if slug not in summary_df.index:
            warnings.warn(f"{model_slug}/{slug} not run yet; left out of the AMCE and WTP tables")
            continue
        status = summary_df.loc[slug, "status"]
        if status != "finished":
            warnings.warn(f"{model_slug}/{slug} is {status}; left out of the AMCE and WTP tables")
            continue
        result = load_paper_run(model_runs_dir / slug)
        run_betas = {}
        for v in VALIDATORS:
            run_betas[v] = result.best_validation[v][1]
        named_betas[f"{summary_df.loc[slug, 'row']}: {slug}"] = run_betas

    kriner_amce, kriner_wtp, grummon_amce = build_amce_and_wtp_tables(named_betas)
    print(f"=== {model_slug} | Kriner 2026 Attribute AMCEs ===")
    display(kriner_amce.round(4))
    print(f"=== {model_slug} | Kriner 2026 Implied Willingness-To-Pay (USD) ===")
    display(kriner_wtp.round(2))
    print(f"=== {model_slug} | Grummon 2025 Food-Label AMCEs ===")
    display(grummon_amce.round(4))

## 4. Figure 1 — KL Anchor Ablation (`row2_wu_table_anchored` vs `row2_ablation_wu_table_no_kl`)

Compares validation `kriner2026/beta_mse`, held-out `grummon2025/beta_mse`, and out-of-domain choice calibration (`grummon2025/kl_loss`) across training steps with ($\alpha = 0.02$) and without ($\alpha = 0.0$) the forward-KL anchor to the base model.

In [ ]:
ABLATION_SLUGS = ("row2_wu_table_anchored", "row2_ablation_wu_table_no_kl")
PANEL_SPECS = [
    ("kriner2026/beta_mse", "Kriner 2026 Beta MSE (Selection)"),
    ("grummon2025/beta_mse", "Grummon 2025 Beta MSE (Held-Out Test)"),
    ("grummon2025/kl_loss", "Grummon 2025 Human KL (Calibration Drift)"),
]

for model_slug in MODEL_SLUGS:
    ablation_runs = []
    for slug in ABLATION_SLUGS:
        if slug not in inventory_by_model[model_slug].index:
            warnings.warn(f"{model_slug}/{slug} not run yet; left out of Figure 1")
            continue
        status = inventory_by_model[model_slug].loc[slug, "status"]
        ablation_runs.append((slug, status))
    if len(ablation_runs) == 0:
        warnings.warn(f"{model_slug}: no Row 2 runs yet; Figure 1 not drawn")
        continue

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for slug, status in ablation_runs:
        result = load_paper_run(PAPER_RUNS_ROOT / model_slug / slug)
        label = f"{slug} (alpha={result.config['kl_anchor_alpha']:g}, {status})"
        for ax, (col, title) in zip(axes, PANEL_SPECS, strict=True):
            # Grummon is validated only every few steps: its other rows are empty
            series = result.val_history_df[col].dropna()
            ax.plot(series.index, series.values, marker="o", markersize=3.5, linewidth=1.6, label=label)
            ax.set_title(f"{model_slug} — {title}", fontsize=10)
            ax.set_xlabel("Optimization Step")
            ax.grid(alpha=0.25)
            ax.legend(fontsize=8.5)
    fig.tight_layout()
    plt.show()

## 5. Pre-Paper Reference Runs (`v8`, `v12`, `v13`, `v14` in `beta-training/runs/`)

Older runs on disk, shown for reference only and never mixed into Tables 1–3. Their designs differ from the paper rows: `v8` and `v12` (Wu + Berke CE) used peak LR `4e-5` and picked best.pt by `kriner2026/val_loss`; `v13` (Wu table only) used no KL anchor ($\alpha = 0$) and 5 epochs; `v14` (CE + population beta loss on Wu + Berke) used peak LR `4e-5`. Their numbers are in the table below.

In [ ]:
LEGACY_SPECS = [
    (
        "kev-0.8b",
        "v13 Wu Table-Only (alpha=0, ep=5)",
        LEGACY_RUNS_ROOT / "train-beta-v13-wu-beta-only-800m" / "beta_only_split_lr=1e-05_S=40_ep=5",
        load_population_run,
    ),
    (
        "kev-0.8b",
        "v8 Full Microdata CE (Wu+Berke, lambda=0)",
        LEGACY_RUNS_ROOT / "train-beta-v8-wu-berke-sample-lrscale-800m" / "lambda=0_lr=4e-05_S=40_ep=1",
        load_run,
    ),
    (
        "kev-4b",
        "v12 Full Microdata CE (Wu+Berke, lambda=0)",
        LEGACY_RUNS_ROOT / "train-beta-v12-wu-berke-scale-4b" / "lambda=0_lr=4e-05_S=40_ep=1",
        load_run,
    ),
    (
        "kev-4b",
        "v14 Full Microdata CE + Pop-Beta (lambda=0.25)",
        LEGACY_RUNS_ROOT / "train-beta-v14-wu-berke-ce-population-4b" / "lambda=0.25_lr=4e-05_S=40_ep=1",
        load_ce_population_run,
    ),
    (
        "kev-4b",
        "v14 Full Microdata CE + Pop-Beta (lambda=1.0)",
        LEGACY_RUNS_ROOT / "train-beta-v14-wu-berke-ce-population-4b" / "lambda=1_lr=4e-05_S=40_ep=1",
        load_ce_population_run,
    ),
]

LEGACY_METRIC_COLUMNS = []
for v in VALIDATORS:
    for metric in ("beta_mse", "beta_mae", "spearman_rho", "kl_loss"):
        LEGACY_METRIC_COLUMNS.append(f"{v}/{metric}")

legacy_rows = []
legacy_betas_by_model: dict[str, dict[str, dict[str, pd.DataFrame]]] = {}

for model_slug, label, run_dir, loader in LEGACY_SPECS:
    if not (run_dir / "run.json").exists():
        warnings.warn(f"{label}: no run.json in {run_dir}; left out")
        continue
    run_info = run_store.read_run_json(run_dir)
    if run_info["status"] != "finished":
        warnings.warn(f"{label} is {run_info['status']}; left out")
        continue

    if model_slug not in legacy_betas_by_model:
        base_info = extract_step0_from_run_dir(run_dir, VALIDATORS)
        legacy_betas_by_model[model_slug] = {"Base LLM (Step 0)": base_info["betas_by_validator"]}
        base_row = {"model": model_slug, "reference_run": "Untrained Base LLM (Step 0)", "status": "step0", "best_step": 0}
        for col in LEGACY_METRIC_COLUMNS:
            base_row[col] = base_info["metrics"][col]
        legacy_rows.append(base_row)

    result = loader(run_dir)
    row = {"model": model_slug, "reference_run": label, "status": run_info["status"], "best_step": int(result.best["step"])}
    for col in LEGACY_METRIC_COLUMNS:
        row[col] = float(result.best_row[col])
    legacy_rows.append(row)
    run_betas = {}
    for v in VALIDATORS:
        run_betas[v] = result.best_validation[v][1]
    legacy_betas_by_model[model_slug][label] = run_betas

if len(legacy_rows) == 0:
    raise FileNotFoundError("None of the reference runs in LEGACY_SPECS is saved and finished")
display(pd.DataFrame(legacy_rows).set_index(["model", "reference_run"]).round(5))
for model_slug, named_betas in legacy_betas_by_model.items():
    _, legacy_wtp, _ = build_amce_and_wtp_tables(named_betas)
    print(f"=== {model_slug} Reference Runs — Kriner 2026 Implied Flight WTP (USD) ===")
    display(legacy_wtp.round(2))